Using Regular XSY Albert based fake news predictor in Renew Test (20%)


Originated from: https://huggingface.co/XSY/albert-base-v2-fakenews-discriminator


In [18]:
import pandas as pd
import torch
from transformers import AlbertTokenizer, AlbertForSequenceClassification
from datasets import Dataset

In [ ]:

df = pd.read_csv("./dataset/renew_combined_test.csv")

df.head()

,author,title,source,date,label,text,quotes
0,Mary Hartel,"""86% of Americans and 82% of gun owners suppor...",Beth Wessel-Kroeschell,12/3/21,True,An effort to strengthen gun owner rights with ...,Statistic: 86% of Americans and 82% of gun own...
1,Yacob Reyes,"""There hasn't been a single of these mass shoo...",Marco Rubio,27/5/2022,False,"Sen. Marco Rubio, R-Fla., said stringent gun r...",There hasn't been a single of these mass shoot...
2,Madison Czopek,"Michigan's Proposal 2 ""will permanently put in...",Ted Nugent,30/9/2022,False,"There are several elements to Proposal 2, a vo...",For all of you Michiganders!!!! Instead of lis...
3,Maria BriceÃÂ±o,"""La presidenta de México Claudia Sheinbaum PRO...",Facebook posts,26/2/2025,False,La orden del presidente Donald Trump a princip...,Es un hecho. La presidenta de México Claudia S...
4,Victoria Knight,"Says Mike Bloomberg is ""a candidate who in 201...",Bernie Sanders,19/2/20,True,U.S. Sen. Bernie Sanders welcomed Democratic r...,We will not defeat Donald Trump with a candida...


In [ ]:
import pandas as pd

df = pd.read_csv("./dataset/renew_combined_test.csv")

print("DataFrame shape:", df.shape)
print("\nSample rows:")
print(df.head())

print("\nNull counts:")
print(df.isnull().sum())

print("\nUnique labels before cleaning:")
print(df["label"].unique())

DataFrame shape: (527, 7)

Sample rows:
             author                                              title  \
0       Mary Hartel  "86% of Americans and 82% of gun owners suppor...   
1       Yacob Reyes  "There hasn't been a single of these mass shoo...   
2    Madison Czopek  Michigan's Proposal 2 "will permanently put in...   
3  Maria BriceÃÂ±o  "La presidenta de México Claudia Sheinbaum PRO...   
4   Victoria Knight  Says Mike Bloomberg is "a candidate who in 201...   

                   source       date  label  \
0  Beth Wessel-Kroeschell    12/3/21   True   
1             Marco Rubio  27/5/2022  False   
2              Ted Nugent  30/9/2022  False   
3          Facebook posts  26/2/2025  False   
4          Bernie Sanders    19/2/20   True   

                                                text  \
0  An effort to strengthen gun owner rights with ...   
1  Sen. Marco Rubio, R-Fla., said stringent gun r...   
2  There are several elements to Proposal 2, a vo...   
3  La or

In [ ]:
df = df[["quotes", "label"]].dropna()

df["label"] = df["label"].map({True: 1, False: 0})

df = df.rename(columns={"quotes": "title"})

print("\nCleaned DataFrame:")
print(df.head())


Cleaned DataFrame:
                                               title  label
0  Statistic: 86% of Americans and 82% of gun own...      1
1  There hasn't been a single of these mass shoot...      0
2  For all of you Michiganders!!!! Instead of lis...      0
3  Es un hecho. La presidenta de México Claudia S...      0
4  We will not defeat Donald Trump with a candida...      1


In [5]:
%pip install sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "albert-base-v2"
)  

def tokenize_fn(batch):
    return tokenizer(batch["title"], padding=True, truncation=True, max_length=128)


encoded_df = df["title"].apply(lambda x: tokenize_fn({"title": x}))

print("\nSample tokenized output:")
print(encoded_df.head())


Sample tokenized output:
0    [input_ids, token_type_ids, attention_mask]
1    [input_ids, token_type_ids, attention_mask]
2    [input_ids, token_type_ids, attention_mask]
3    [input_ids, token_type_ids, attention_mask]
4    [input_ids, token_type_ids, attention_mask]
Name: title, dtype: object


In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(df)

print("\nHugging Face Dataset:")
print(hf_dataset)


Hugging Face Dataset:
Dataset({
    features: ['title', 'label'],
    num_rows: 527
})


In [ ]:
from torch.utils.data import DataLoader

batch_size = 16

train_dataloader = DataLoader(hf_dataset, batch_size=batch_size, shuffle=True)

for batch in train_dataloader:
    print(batch)
    break  

{'title': ['I beat her. It\'s easier when you win, and they all said, \'Lock her up.\' And I felt I could have done it, but I felt it would have been a terrible thing. Hillary Clinton, I didn\'t say, \'Lock her up,\' but the people would all say, \'Lock her up. Lock her up.\' I said, pretty openly. I say all right, come on. Just relax. Let\'s go. We got to make our country great. I will say this, Hillary Clinton has to go to jail, OK? She has to go to jail. \x85 She is guilty as hell. I think she should be in jail for what she did with her emails. Because you\'d be in jail. She deleted the emails. She has to go to jail. For what she\'s done, they should lock her up, she\'s disgraceful. Hillary Clinton should have been prosecuted and should be in jail. Instead she is running for president in what looks like a rigged election. If she were to win this election, it would create an unprecedented constitutional crisis. In that situation, we could very well have a sitting president under felo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "XSY/albert-base-v2-fakenews-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
import torch
from torch.utils.data import TensorDataset

encodings = tokenizer.batch_encode_plus(
    df["title"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt",
)

labels = torch.tensor(df["label"].tolist())

input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)
labels = labels.to(device)

dataset = TensorDataset(input_ids, attention_mask, labels)
dataloader = DataLoader(dataset, batch_size=16)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask, labels = batch

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"\nAccuracy on custom dataset: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["FAKE", "REAL"]))


Accuracy on custom dataset: 0.4156

Classification Report:
              precision    recall  f1-score   support

        FAKE       0.42      0.35      0.38       273
        REAL       0.41      0.49      0.45       254

    accuracy                           0.42       527
   macro avg       0.42      0.42      0.41       527
weighted avg       0.42      0.42      0.41       527



Fine tuning on LIAR + R to test with the Renew combined test dataset


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import pandas as pd
import torch


In [ ]:
df = pd.read_csv("./dataset/LIAR_Renew_training.csv")

df["label"] = df["label"].astype(int)

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


In [ ]:
model_checkpoint = "XSY/albert-base-v2-fakenews-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(example):
    return tokenizer(example["statement"], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/1896 [00:00<?, ? examples/s]

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)


In [ ]:
pip install evaluate

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./albert_finetuned_renew",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

c:\Users\alber\anaconda3\envs\ai-testing\lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\alber\AppData\Local\Temp\ipykernel_24188\1744526114.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.533400,0.413822,0.829384


TrainOutput(global_step=119, training_loss=0.5253333604636312, metrics={'train_runtime': 3554.5521, 'train_samples_per_second': 0.533, 'train_steps_per_second': 0.033, 'total_flos': 45310800936960.0, 'train_loss': 0.5253333604636312, 'epoch': 1.0})

In [ ]:
model_save_path = "./fine_tuned_LIAR_Renew_albert"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to: {model_save_path}")


Model and tokenizer saved to: ./fine_tuned_LIAR_Renew_albert


Testing the Renew + R, in the renew combined test.csv file


In [ ]:
import pandas as pd

test_path = './dataset/renew_combined_test.csv'
test_df = pd.read_csv(test_path)

test_df = test_df[['label', 'quotes']]

test_df = test_df.rename(columns={'quotes': 'statement'})

test_df['label'] = test_df['label'].astype(str).str.strip().str.upper()

test_df['label'] = test_df['label'].map({'TRUE': 1, 'FALSE': 0})

print(test_df.head())

   label                                          statement
0      1  Statistic: 86% of Americans and 82% of gun own...
1      0  There hasn't been a single of these mass shoot...
2      0  For all of you Michiganders!!!! Instead of lis...
3      0  Es un hecho. La presidenta de México Claudia S...
4      1  We will not defeat Donald Trump with a candida...


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

model_path = "./fine_tuned_LIAR_Renew_albert"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval() 

test_dataset = Dataset.from_pandas(test_df)

In [ ]:

def tokenize_function(examples):
    return tokenizer(examples["statement"], padding=True, truncation=True, max_length=512)

test_dataset = test_dataset.map(tokenize_function, batched=True)

test_dataset = test_dataset.rename_column("label", "labels")

test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

test_loader = DataLoader(test_dataset, batch_size=8)

Map:   0%|          | 0/527 [00:00<?, ? examples/s]

In [ ]:
predictions = []
true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(labels.tolist())

from sklearn.metrics import classification_report

print(classification_report(true_labels, predictions, target_names=["False", "True"]))

100%|██████████| 66/66 [03:04<00:00,  2.80s/it]

              precision    recall  f1-score   support

       False       0.79      0.74      0.77       273
        True       0.74      0.79      0.76       254

    accuracy                           0.76       527
   macro avg       0.77      0.77      0.76       527
weighted avg       0.77      0.76      0.76       527



Testing the default model performance over the default testing dataset


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

politifact_fake_df = pd.read_csv("./Politifact dataset/politifact kaggle fake.csv")
politifact_true_df = pd.read_csv("./Politifact dataset/politifact kaggle true.csv")

print("Fake News Samples:")
print(politifact_fake_df.head())

print("\nTrue News Samples:")
print(politifact_true_df.head())


Fake News Samples:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  

True News Samples:
                                               title  \
0  As U.S. budget fight looms, Republicans fl

In [ ]:
politifact_fake_df["label"] = 0
politifact_true_df["label"] = 1

politifact_combined_df = pd.concat([politifact_fake_df, politifact_true_df], ignore_index=True)

politifact_combined_df = politifact_combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(politifact_combined_df.head())

                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text       subject  \
0  21st Century Wire says Ben Stein, reputable pr...       US_News   
1  WASHINGTON (Reuters) - U.S. President Donald T...  politicsNews   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...  politicsNews   
3  On Monday, Donald Trump once again embarrassed...          News   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...  politicsNews   

                  date  label  
0    February 13, 2017      0  
1       April 5, 2017       1  
2  September 27, 2017       1  
3         May 22, 2017      0  
4       June 24, 2016       1  


In [ ]:

train_df, test_df = train_test_split(
    politifact_combined_df, 
    test_size=0.2, 
    random_state=42,
    stratify=politifact_combined_df["label"]
)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")
politifact_copy_df = test_df.copy()

Training samples: 35918
Testing samples: 8980


In [ ]:
politifact_copy_df = test_df.copy()

politifact_copy_df = politifact_copy_df.rename(columns={"text": "statement"})

politifact_statements = politifact_copy_df['statement'].tolist()
politifact_true_labels = politifact_copy_df['label'].tolist()

print(politifact_copy_df.head())

                                                   title  \
14864  Philippines doctor linked to New York attack p...   
7865   More Republicans expect Clinton, rather than T...   
16749  DEMOCRAT UNDERBELLY EXPOSED: Out-Of-Control Vi...   
29452  Those who boycott Syrian congress may be sidel...   
44890  Burundi opposition platform boycotts new round...   

                                               statement       subject  \
14864  MANILA (Reuters) - A Filipino accused by the U...     worldnews   
7865   NEW YORK (Reuters) - More Republicans now thin...  politicsNews   
16749  Donald Trump on Wednesday slammed what he desc...      politics   
29452  ASTANA (Reuters) - Syrian groups who choose to...     worldnews   
44890  NAIROBI (Reuters) - Burundi s main opposition ...     worldnews   

                     date  label  
14864   October 10, 2017       1  
7865    October 26, 2016       1  
16749        May 25, 2016      0  
29452   October 31, 2017       1  
44890  November 28,

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model_name = "XSY/albert-base-v2-fakenews-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

AlbertForSequenceClassification(
  (albert): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(30000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertSdpaAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=

In [ ]:
politifact_copy_df = politifact_copy_df.copy()

politifact_copy_df = politifact_copy_df[['statement', 'label']]

politifact_copy_df['label'] = politifact_copy_df['label'].apply(lambda x: 1 if x == 'TRUE' or x == 'mostly-true' else 0)

encodings = tokenizer.batch_encode_plus(
    politifact_copy_df["statement"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

labels = torch.tensor(politifact_copy_df["label"].tolist())

input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)
labels = labels.to(device)

dataset = TensorDataset(input_ids, attention_mask, labels)
dataloader = DataLoader(dataset, batch_size=16)

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask, labels = batch
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)

tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp)

print(f"Accuracy on Politifact dataset: {accuracy:.4f}")
print(f"Precision on Politifact dataset: {precision:.4f}")
print(f"Recall on Politifact dataset: {recall:.4f}")
print(f"Specificity on Politifact dataset: {specificity:.4f}")
print(f"F1 Score on Politifact dataset: {f1:.4f}")
print("Confusion Matrix (Politifact dataset):")
print(conf_matrix)

Accuracy on Politifact dataset: 0.9398
Precision on Politifact dataset: 0.0000
Recall on Politifact dataset: 0.0000
Specificity on Politifact dataset: 0.9398
F1 Score on Politifact dataset: 0.0000
Confusion Matrix (Politifact dataset):
[[8439  541]
 [   0    0]]


c:\Users\alber\anaconda3\envs\ai-testing\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fine tuning process over Renew Training


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "XSY/albert-base-v2-fakenews-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
import pandas as pd

train_df = pd.read_csv("./ReNew dataset/combined renew train.csv")

train_df.head()

,label,statement
0,1,Wisconsin's archaic abortion ban is older than...
1,0,Trump can't vote for himself in the November e...
2,0,Putin bombs Biden-owned villa in Ukraine while...
3,0,While Wanggaard testified that 'this is very n...
4,1,If you're a low-wage worker and you're single ...


In [ ]:
train_encodings = tokenizer.batch_encode_plus(
    train_df["statement"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

import torch

train_labels = torch.tensor(train_df["label"].tolist())

In [ ]:
from torch.utils.data import Dataset

class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(train_df, test_size=0.1, random_state=42)

train_encodings = tokenizer.batch_encode_plus(
    train_df["statement"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

train_labels = torch.tensor(train_df["label"].tolist())

valid_encodings = tokenizer.batch_encode_plus(
    valid_df["statement"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

valid_labels = torch.tensor(valid_df["label"].tolist())

train_dataset = NewsDataset(train_encodings, train_labels)
valid_dataset = NewsDataset(valid_encodings, valid_labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fine_tuned_models/albert_finetuned_fake_news",  
    num_train_epochs=3,  
    per_device_train_batch_size=16,  
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',  
    logging_steps=10,
    evaluation_strategy="epoch",  
    save_strategy="epoch",  
    load_best_model_at_end=True
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset  
)

trainer.train()


C:\Users\alber\AppData\Local\Temp\ipykernel_23352\2634602142.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss
1,0.653900,0.668958
2,0.570500,0.580828
3,0.479700,0.546418


C:\Users\alber\AppData\Local\Temp\ipykernel_23352\2634602142.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
C:\Users\alber\AppData\Local\Temp\ipykernel_23352\2634602142.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
C:\Users\alber\AppData\Local\Temp\ipykernel_23352\2634602142.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val i

TrainOutput(global_step=357, training_loss=0.8628863520315048, metrics={'train_runtime': 470.9816, 'train_samples_per_second': 12.058, 'train_steps_per_second': 0.758, 'total_flos': 33929329973760.0, 'train_loss': 0.8628863520315048, 'epoch': 3.0})

In [ ]:
output_dir = "./fine tuned models/albert politifact fine tuned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)


('./fine tuned models/albert politifact fine tuned\\tokenizer_config.json',
 './fine tuned models/albert politifact fine tuned\\special_tokens_map.json',
 './fine tuned models/albert politifact fine tuned\\tokenizer.json')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

politifact_fake_df = pd.read_csv("./Politifact dataset/politifact kaggle fake.csv")
politifact_true_df = pd.read_csv("./Politifact dataset/politifact kaggle true.csv")

print("Fake News Samples:")
print(politifact_fake_df.head())

print("\nTrue News Samples:")
print(politifact_true_df.head())


,statement,label
28661,UNITED NATIONS (Reuters) - Russia s U.N. Ambas...,0
39091,At least two Republican campaigns are using ve...,0
17023,"During an interview on CNN Sunday morning, Jak...",0
37544,WASHINGTON (Reuters) - Longtime Trump communic...,0
9615,Hillary Clinton is known world-wide for saying...,0
31238,For all the hate Democrats openly express for ...,0
31974,"If Oklahoma Republicans have their way, the su...",0
3819,When a bunch of kids were recently asked what ...,0
24240,OTTAWA (Reuters) - Canada said on Wednesday th...,0
24018,DAKAR/ACCRA (Reuters) - The International Trib...,0


In [ ]:
politifact_fake_df["label"] = 0
politifact_true_df["label"] = 1

politifact_combined_df = pd.concat([politifact_fake_df, politifact_true_df], ignore_index=True)

politifact_combined_df = politifact_combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(politifact_combined_df.head())

                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text       subject  \
0  21st Century Wire says Ben Stein, reputable pr...       US_News   
1  WASHINGTON (Reuters) - U.S. President Donald T...  politicsNews   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...  politicsNews   
3  On Monday, Donald Trump once again embarrassed...          News   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...  politicsNews   

                  date  label  
0    February 13, 2017      0  
1       April 5, 2017       1  
2  September 27, 2017       1  
3         May 22, 2017      0  
4       June 24, 2016       1  


In [ ]:
train_df, test_df = train_test_split(
    politifact_combined_df, 
    test_size=0.2, 
    random_state=42,
    stratify=politifact_combined_df["label"]
)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

politifact_copy_df = test_df.copy()
politifact_copy_df = politifact_copy_df.rename(columns={"text": "statement"})

politifact_statements = politifact_copy_df['statement'].tolist()
politifact_true_labels = politifact_copy_df['label'].tolist()

print(politifact_copy_df.head())

Training samples: 35918
Testing samples: 8980
                                                   title  \
14864  Philippines doctor linked to New York attack p...   
7865   More Republicans expect Clinton, rather than T...   
16749  DEMOCRAT UNDERBELLY EXPOSED: Out-Of-Control Vi...   
29452  Those who boycott Syrian congress may be sidel...   
44890  Burundi opposition platform boycotts new round...   

                                               statement       subject  \
14864  MANILA (Reuters) - A Filipino accused by the U...     worldnews   
7865   NEW YORK (Reuters) - More Republicans now thin...  politicsNews   
16749  Donald Trump on Wednesday slammed what he desc...      politics   
29452  ASTANA (Reuters) - Syrian groups who choose to...     worldnews   
44890  NAIROBI (Reuters) - Burundi s main opposition ...     worldnews   

                     date  label  
14864   October 10, 2017       1  
7865    October 26, 2016       1  
16749        May 25, 2016      0  
29452   

In [ ]:
df_label_1 = politifact_copy_df[politifact_copy_df['label'] == 1]

print(df_label_1.head())

                                                   title  \
14864  Philippines doctor linked to New York attack p...   
7865   More Republicans expect Clinton, rather than T...   
29452  Those who boycott Syrian congress may be sidel...   
44890  Burundi opposition platform boycotts new round...   
5526   U.S. representatives question bank regulator o...   

                                               statement       subject  \
14864  MANILA (Reuters) - A Filipino accused by the U...     worldnews   
7865   NEW YORK (Reuters) - More Republicans now thin...  politicsNews   
29452  ASTANA (Reuters) - Syrian groups who choose to...     worldnews   
44890  NAIROBI (Reuters) - Burundi s main opposition ...     worldnews   
5526   WASHINGTON (Reuters) - Leaders of the congress...  politicsNews   

                     date  label  
14864   October 10, 2017       1  
7865    October 26, 2016       1  
29452   October 31, 2017       1  
44890  November 28, 2017       1  
5526        June 9,

In [ ]:
encodings = tokenizer.batch_encode_plus(
    politifact_copy_df["statement"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

In [ ]:
labels = torch.tensor(politifact_copy_df["label"].tolist())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)
labels = labels.to(device)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(input_ids, attention_mask, labels)
dataloader = DataLoader(dataset, batch_size=16)

In [ ]:
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask, labels = batch

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)

tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp)

print(f"Accuracy on Politifact dataset: {accuracy:.4f}")
print(f"Precision on Politifact dataset: {precision:.4f}")
print(f"Recall on Politifact dataset: {recall:.4f}")
print(f"Specificity on Politifact dataset: {specificity:.4f}")
print(f"F1 Score on Politifact dataset: {f1:.4f}")
print("Confusion Matrix (Politifact dataset):")
print(conf_matrix)


Accuracy on Politifact dataset: 0.7384
Precision on Politifact dataset: 0.7735
Recall on Politifact dataset: 0.6387
Specificity on Politifact dataset: 0.8294
F1 Score on Politifact dataset: 0.6997
Confusion Matrix (Politifact dataset):
[[3895  801]
 [1548 2736]]


Testing the Politifact + Renew fine tuned version with the renew testing dataset


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model_path = "./fine tuned models/albert politifact fine tuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

AlbertForSequenceClassification(
  (albert): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(30000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertSdpaAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=

In [ ]:
import pandas as pd

test_df = pd.read_csv('./ReNew dataset/combined renew test.csv')

test_df = test_df[['label', 'statement']]

test_encodings = tokenizer.batch_encode_plus(
    test_df["statement"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

test_labels = torch.tensor(test_df["label"].tolist())

input_ids = test_encodings["input_ids"].to(device)
attention_mask = test_encodings["attention_mask"].to(device)
test_labels = test_labels.to(device)

test_dataset = TensorDataset(input_ids, attention_mask, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=16)


In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        input_ids, attention_mask, labels = batch

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)

tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


Accuracy: 0.7818
Precision: 0.7251
Recall: 0.9091
Specificity: 0.6540
F1 Score: 0.8067
Confusion Matrix:
[[172  91]
 [ 24 240]]


Testing the Politifact + Renew fine tuned version with the Politifact testing


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model_path = "./fine tuned models/albert politifact fine tuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

AlbertForSequenceClassification(
  (albert): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(30000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertSdpaAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=

In [ ]:
import pandas as pd

test_df = pd.read_csv('./Politifact dataset/merged/merged_politifact_test.csv')

if 'text' in test_df.columns:
    test_df.rename(columns={'text': 'statement'}, inplace=True)

test_df = test_df[['label', 'statement']]

test_encodings = tokenizer.batch_encode_plus(
    test_df["statement"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

test_labels = torch.tensor(test_df["label"].tolist())

input_ids = test_encodings["input_ids"].to(device)
attention_mask = test_encodings["attention_mask"].to(device)
test_labels = test_labels.to(device)

test_dataset = TensorDataset(input_ids, attention_mask, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=16)


In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        input_ids, attention_mask, labels = batch

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)

tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.7384
Precision: 0.7735
Recall: 0.6387
Specificity: 0.8294
F1 Score: 0.6997
Confusion Matrix:
[[3895  801]
 [1548 2736]]
